In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.fft as fft
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from scipy.ndimage import gaussian_filter, binary_erosion, binary_dilation
from skimage import morphology
from tqdm import tqdm
import os
import warnings
warnings.filterwarnings('ignore')

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# ================== Step 1: Custom Dataset for Fruits-360 ==================

class FruitsDataset(Dataset):
    """Custom dataset for Fruits-360"""
    
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.classes = sorted([d for d in os.listdir(root_dir) 
                              if os.path.isdir(os.path.join(root_dir, d))])
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}
        
        self.samples = []
        for class_name in self.classes:
            class_dir = os.path.join(root_dir, class_name)
            for img_name in os.listdir(class_dir):
                if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                    self.samples.append((os.path.join(class_dir, img_name), 
                                       self.class_to_idx[class_name]))
        
        print(f"Found {len(self.samples)} images in {len(self.classes)} classes")
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
        
        return image, label

def load_fruits_dataset(data_root):
    """Load Fruits-360 dataset with augmentation"""
    
    # Enhanced augmentation for better generalization
    transform_train = transforms.Compose([
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.3),
        transforms.RandomRotation(20),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    train_dir = os.path.join(data_root, 'Training')
    test_dir = os.path.join(data_root, 'Test')
    
    trainset = FruitsDataset(train_dir, transform=transform_train)
    testset = FruitsDataset(test_dir, transform=transform_test)
    
    return trainset, testset, trainset.classes

# ================== Step 2: FFT Conversion Functions ==================

def spatial_to_frequency(images):
    """Convert spatial domain images to frequency domain using FFT"""
    freq_complex = fft.fft2(images, dim=(-2, -1))
    freq_complex = fft.fftshift(freq_complex, dim=(-2, -1))
    
    freq_magnitude = torch.abs(freq_complex)
    freq_phase = torch.angle(freq_complex)
    
    # Log-scale normalization for magnitude
    eps = torch.mean(freq_magnitude) * 0.01
    freq_magnitude_log = torch.log(freq_magnitude + eps)
    freq_magnitude_normalized = (freq_magnitude_log - freq_magnitude_log.mean()) / (freq_magnitude_log.std() + 1e-8)
    
    # Encode phase as cosine and sine
    phase_cos = torch.cos(freq_phase)
    phase_sin = torch.sin(freq_phase)
    
    # Concatenate magnitude and phase information (9 channels total)
    freq_features = torch.cat([freq_magnitude_normalized, phase_cos, phase_sin], dim=1)
    
    return freq_features, freq_phase, freq_complex

def frequency_to_spatial(freq_magnitude, freq_phase):
    """Convert frequency domain back to spatial domain"""
    freq_magnitude = torch.exp(freq_magnitude)
    freq_complex = freq_magnitude * torch.exp(1j * freq_phase)
    
    freq_complex = fft.ifftshift(freq_complex, dim=(-2, -1))
    spatial_complex = fft.ifft2(freq_complex, dim=(-2, -1))
    spatial_images = torch.real(spatial_complex)
    
    return spatial_images

# ================== Step 3: Frequency Domain Dataset ==================

class FrequencyDomainDataset(Dataset):
    """Custom dataset for frequency domain representations"""
    
    def __init__(self, original_dataset):
        self.original_dataset = original_dataset
        
    def __len__(self):
        return len(self.original_dataset)
    
    def __getitem__(self, idx):
        image, label = self.original_dataset[idx]
        
        # Convert to frequency domain
        freq_features, freq_phase, _ = spatial_to_frequency(image.unsqueeze(0))
        freq_features = freq_features.squeeze(0)
        freq_phase = freq_phase.squeeze(0)
        
        return freq_features, label, freq_phase

# ================== Step 4: Enhanced CNN Model for Frequency Domain ==================

class FrequencyDomainCNN(nn.Module):
    """Enhanced ResNet50-based model for frequency domain (100x100 images)"""
    
    def __init__(self, num_classes, dropout_rate=0.5):
        super(FrequencyDomainCNN, self).__init__()
        
        # Use ResNet50 for better feature extraction
        self.model = models.resnet50(pretrained=True)
        
        # Modify first conv layer for 9-channel input
        self.model.conv1 = nn.Conv2d(9, 64, kernel_size=7, stride=2, padding=3, bias=False)
        
        # Enhanced classifier head with more capacity
        num_features = self.model.fc.in_features
        self.model.fc = nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(num_features, 1024),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(1024),
            nn.Dropout(dropout_rate * 0.7),
            nn.Linear(1024, 512),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(512),
            nn.Dropout(dropout_rate * 0.5),
            nn.Linear(512, num_classes)
        )
        
        self._initialize_weights()
    
    def _initialize_weights(self):
        """Initialize new layers with proper weights"""
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, (nn.BatchNorm2d, nn.BatchNorm1d)):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
    
    def forward(self, x):
        return self.model(x)
    
    def get_activations(self, x):
        """Extract feature maps from last conv layer before global pooling"""
        x = self.model.conv1(x)
        x = self.model.bn1(x)
        x = self.model.relu(x)
        x = self.model.maxpool(x)
        
        x = self.model.layer1(x)
        x = self.model.layer2(x)
        x = self.model.layer3(x)
        x = self.model.layer4(x)
        
        return x

# ================== Step 5: Training Functions ==================

class EarlyStopping:
    """Early stopping to prevent overfitting"""
    def __init__(self, patience=10, min_delta=0.0, verbose=True):
        self.patience = patience
        self.min_delta = min_delta
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.best_model_state = None
        
    def __call__(self, val_accuracy, model):
        score = val_accuracy
        
        if self.best_score is None:
            self.best_score = score
            self.best_model_state = model.state_dict()
        elif score < self.best_score + self.min_delta:
            self.counter += 1
            if self.verbose:
                print(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.best_model_state = model.state_dict()
            self.counter = 0

def train_model(model, train_loader, val_loader, epochs=50, lr=0.001, weight_decay=1e-4):
    """Train the frequency domain CNN model"""
    
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    
    # Use different learning rates for pretrained and new layers
    pretrained_params = []
    new_params = []
    
    for name, param in model.named_parameters():
        if 'model.fc' in name or 'model.conv1' in name:
            new_params.append(param)
        else:
            pretrained_params.append(param)
    
    optimizer = torch.optim.AdamW([
        {'params': pretrained_params, 'lr': lr * 0.1},
        {'params': new_params, 'lr': lr}
    ], weight_decay=weight_decay)
    
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=10, T_mult=2, eta_min=1e-6
    )
    
    early_stopping = EarlyStopping(patience=15, min_delta=0.1, verbose=True)
    
    train_losses = []
    val_losses = []
    train_accuracies = []
    val_accuracies = []
    
    best_val_accuracy = 0.0
    best_model_state = None
    
    for epoch in range(epochs):
        # Training phase
        model.train()
        running_loss = 0.0
        correct_train = 0
        total_train = 0
        
        train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]")
        for i, (freq_images, labels, _) in enumerate(train_pbar):
            freq_images, labels = freq_images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(freq_images)
            loss = criterion(outputs, labels)
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total_train += labels.size(0)
            correct_train += (predicted == labels).sum().item()
            
            train_pbar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'acc': f'{100 * correct_train / total_train:.2f}%'
            })
        
        scheduler.step()
        
        avg_train_loss = running_loss / len(train_loader)
        train_accuracy = 100 * correct_train / total_train
        train_losses.append(avg_train_loss)
        train_accuracies.append(train_accuracy)
        
        # Validation phase
        model.eval()
        running_val_loss = 0.0
        correct = 0
        total = 0
        
        with torch.no_grad():
            for freq_images, labels, _ in tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]"):
                freq_images, labels = freq_images.to(device), labels.to(device)
                outputs = model(freq_images)
                loss = criterion(outputs, labels)
                running_val_loss += loss.item()
                
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
        
        avg_val_loss = running_val_loss / len(val_loader)
        val_accuracy = 100 * correct / total
        val_losses.append(avg_val_loss)
        val_accuracies.append(val_accuracy)
        
        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            best_model_state = model.state_dict()
        
        print(f'\nEpoch [{epoch+1}/{epochs}]')
        print(f'Train Loss: {avg_train_loss:.4f}, Train Acc: {train_accuracy:.2f}%')
        print(f'Val Loss: {avg_val_loss:.4f}, Val Acc: {val_accuracy:.2f}%')
        print(f'Learning Rate: {optimizer.param_groups[0]["lr"]:.6f}\n')
        
        early_stopping(val_accuracy, model)
        if early_stopping.early_stop:
            print("Early stopping triggered!")
            break
    
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        print(f"\nLoaded best model with validation accuracy: {best_val_accuracy:.2f}%")
    
    return train_losses, val_losses, train_accuracies, val_accuracies

# ================== Step 6: FIXED Score-CAM Implementation ==================

class ImprovedScoreCAM:
    """Score-CAM implementation matching CIFAR-10 quality"""
    
    def __init__(self, model):
        self.model = model
        self.model.eval()
        
    def generate_cam(self, input_image, target_class, batch_size=32):
        """Generate Score-CAM with proper normalization"""
        activations = self.model.get_activations(input_image)
        b, k, h, w = activations.shape
        
        # Get base score
        with torch.no_grad():
            base_output = self.model(input_image)
            base_score = F.softmax(base_output, dim=1)[0, target_class].item()
        
        _, _, input_h, input_w = input_image.shape
        
        # Upsample activations to input size
        upsampled_activations = F.interpolate(
            activations, 
            size=(input_h, input_w), 
            mode='bilinear', 
            align_corners=False
        )
        
        upsampled_activations = upsampled_activations.squeeze(0)
        
        weights = []
        
        # Process in batches for efficiency
        for i in range(0, k, batch_size):
            batch_end = min(i + batch_size, k)
            batch_activations = upsampled_activations[i:batch_end]
            
            batch_weights = []
            for act_map in batch_activations:
                # Normalize activation map to [0, 1]
                act_min = act_map.min()
                act_max = act_map.max()
                
                if act_max > act_min:
                    act_map_norm = (act_map - act_min) / (act_max - act_min)
                else:
                    act_map_norm = torch.zeros_like(act_map)
                
                # Apply activation map as mask
                masked_input = input_image * act_map_norm.unsqueeze(0).unsqueeze(0)
                
                # Get score for masked input
                with torch.no_grad():
                    output = self.model(masked_input)
                    score = F.softmax(output, dim=1)[0, target_class].item()
                
                # Weight is the increase in score
                weight = max(0, score - base_score * 0.1)
                batch_weights.append(weight)
            
            weights.extend(batch_weights)
        
        weights = torch.FloatTensor(weights).to(device)
        
        # Normalize weights
        if weights.sum() > 0:
            weights = weights / weights.sum()
        
        # Compute weighted combination of activation maps
        activations_2d = activations.squeeze(0)
        cam = torch.zeros((h, w), dtype=torch.float32).to(device)
        
        for i, w in enumerate(weights):
            cam += w * activations_2d[i]
        
        # Apply ReLU
        cam = F.relu(cam)
        
        # Normalize to [0, 1]
        if cam.max() > 0:
            cam = cam / cam.max()
        
        # Upsample to input size
        cam = F.interpolate(
            cam.unsqueeze(0).unsqueeze(0),
            size=(input_h, input_w),
            mode='bilinear',
            align_corners=False
        ).squeeze()
        
        return cam.cpu().detach().numpy(), weights.cpu().detach().numpy()

# ================== Step 7: FIXED Spatial Domain Mapping ==================

def apply_improved_scorecam_mapping(model, freq_image, phase, target_class, original_image):
    """Apply Score-CAM with better spatial mapping - matching CIFAR-10 quality"""
    
    model.eval()
    freq_input = freq_image.clone().detach().to(device)
    
    # Generate Score-CAM in frequency domain
    scorecam = ImprovedScoreCAM(model)
    cam_freq, weights = scorecam.generate_cam(freq_input, target_class, batch_size=32)
    
    # Denormalize original image
    if original_image.dim() == 4:
        original_image = original_image.squeeze(0)
    
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    original_denorm = original_image.cpu() * std + mean
    original_denorm = torch.clamp(original_denorm, 0, 1)
    original_np = original_denorm.permute(1, 2, 0).numpy()
    
    # Create saliency map with improved processing
    saliency_map = create_cifar_quality_saliency(cam_freq, original_np)
    
    # Create highlighted visualization
    highlighted = create_cifar_quality_overlay(original_np, saliency_map)
    
    return None, saliency_map, highlighted, original_np, cam_freq

def create_cifar_quality_saliency(cam_freq, original_image):
    """Create saliency map matching CIFAR-10 quality"""
    
    # Resize cam if needed
    h, w = original_image.shape[:2]
    if cam_freq.shape != (h, w):
        from scipy.ndimage import zoom
        zoom_factors = (h / cam_freq.shape[0], w / cam_freq.shape[1])
        cam_resized = zoom(cam_freq, zoom_factors, order=3)
    else:
        cam_resized = cam_freq.copy()
    
    # Apply moderate Gaussian smoothing
    saliency = gaussian_filter(cam_resized, sigma=1.5)
    
    # Normalize to [0, 1]
    if saliency.max() > saliency.min():
        saliency = (saliency - saliency.min()) / (saliency.max() - saliency.min())
    
    # Use LOWER threshold for better object coverage (like CIFAR-10)
    threshold = np.percentile(saliency, 40)  # Changed from 70 to 40
    
    # Create binary mask
    binary_mask = saliency > threshold
    
    # Apply minimal morphological operations
    if binary_mask.sum() > 0:
        # Remove very small noise
        binary_mask = morphology.remove_small_objects(binary_mask, min_size=20)
        
        # Fill small holes
        binary_mask = morphology.remove_small_holes(binary_mask, area_threshold=30)
        
        # Very light dilation for smooth edges
        kernel = morphology.disk(1)
        binary_mask = morphology.binary_dilation(binary_mask, kernel)
    
    # Apply mask to saliency
    saliency_masked = saliency * binary_mask
    
    # Apply edge-aware smoothing
    saliency_final = edge_preserving_smooth(saliency_masked, original_image)
    
    # Final normalization
    if saliency_final.max() > 0:
        saliency_final = (saliency_final - saliency_final.min()) / (saliency_final.max() - saliency_final.min())
    
    # Enhance contrast slightly
    saliency_final = np.power(saliency_final, 0.8)
    
    return saliency_final

def edge_preserving_smooth(saliency, guide_image, sigma_spatial=3, sigma_range=0.1):
    """Apply bilateral-like filtering for edge preservation"""
    try:
        from scipy.ndimage import gaussian_filter
        
        # Convert guide to grayscale
        if len(guide_image.shape) == 3:
            guide_gray = np.mean(guide_image, axis=2)
        else:
            guide_gray = guide_image
        
        # Compute spatial weights
        result = gaussian_filter(saliency, sigma=sigma_spatial)
        
        # Simple edge-aware adjustment
        edges = np.abs(gaussian_filter(guide_gray, sigma=1) - guide_gray)
        edges = (edges - edges.min()) / (edges.max() - edges.min() + 1e-8)
        
        # Preserve edges while smoothing
        result = result * (1 - edges * 0.3) + saliency * (edges * 0.3)
        
        return result
    except:
        return saliency

def create_cifar_quality_overlay(original_image, saliency_map, alpha=0.5):
    """Create overlay matching CIFAR-10 visualization quality"""
    
    # Apply jet colormap
    saliency_colored = plt.cm.jet(saliency_map)[:, :, :3]
    
    # Create smooth alpha channel based on saliency intensity
    alpha_channel = saliency_map ** 0.7
    alpha_channel = gaussian_filter(alpha_channel, sigma=0.5)
    alpha_channel = alpha_channel[:, :, np.newaxis]
    
    # Blend with original image
    highlighted = (1 - alpha * alpha_channel) * original_image + alpha * alpha_channel * saliency_colored
    highlighted = np.clip(highlighted, 0, 1)
    
    return highlighted

# ================== Step 8: Visualization Functions ==================

def plot_scorecam_results(original_np, freq_magnitude, cam_freq, saliency_map, 
                          highlighted, prediction, true_label, classes, confidence):
    """Plot comprehensive Score-CAM results"""
    
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    fig.suptitle(f'Frequency Domain CNN - Enhanced Score-CAM Explainability (Fruits-360)', 
                 fontsize=16, fontweight='bold', y=0.995)
    
    # Original Image
    axes[0, 0].imshow(original_np)
    axes[0, 0].set_title(f'Original Image\nGround Truth: {classes[true_label]}', 
                         fontsize=11, fontweight='bold')
    axes[0, 0].axis('off')
    
    # Frequency Domain
    freq_display = freq_magnitude[:, :3, :, :].squeeze(0).mean(0).cpu().numpy()
    im1 = axes[0, 1].imshow(freq_display, cmap='viridis')
    axes[0, 1].set_title('Frequency Domain\n(Magnitude Spectrum)', 
                         fontsize=11, fontweight='bold')
    axes[0, 1].axis('off')
    plt.colorbar(im1, ax=axes[0, 1], fraction=0.046, pad=0.04)
    
    # Score-CAM in Frequency Domain
    im2 = axes[0, 2].imshow(cam_freq, cmap='jet')
    axes[0, 2].set_title('Score-CAM\n(Frequency Domain)', fontsize=11, fontweight='bold')
    axes[0, 2].axis('off')
    plt.colorbar(im2, ax=axes[0, 2], fraction=0.046, pad=0.04)
    
    # Prediction
    correct = "✓" if prediction == true_label else "✗"
    color = 'green' if prediction == true_label else 'red'
    axes[0, 3].text(0.5, 0.5, f'{correct} Prediction:\n{classes[prediction]}\n\nConfidence:\n{confidence:.1f}%', 
                    ha='center', va='center', fontsize=13, fontweight='bold',
                    bbox=dict(boxstyle='round', facecolor=color, alpha=0.3))
    axes[0, 3].set_title('Model Prediction', fontsize=11, fontweight='bold')
    axes[0, 3].axis('off')
    
    # Saliency Map (Mapped to Spatial)
    im3 = axes[1, 0].imshow(saliency_map, cmap='hot')
    axes[1, 0].set_title('Saliency Map\n(Mapped to Spatial)', fontsize=11, fontweight='bold')
    axes[1, 0].axis('off')
    plt.colorbar(im3, ax=axes[1, 0], fraction=0.046, pad=0.04)
    
    # Highlighted Regions (Overlay on Original)
    axes[1, 1].imshow(highlighted)
    axes[1, 1].set_title('Highlighted Regions\n(Overlay on Original)', 
                         fontsize=11, fontweight='bold')
    axes[1, 1].axis('off')
    
    # Importance Heatmap (50% overlay)
    axes[1, 2].imshow(original_np)
    axes[1, 2].imshow(saliency_map, cmap='jet', alpha=0.5)
    axes[1, 2].set_title('Importance Heatmap\n(50% Overlay)', fontsize=11, fontweight='bold')
    axes[1, 2].axis('off')
    
    # Before/After Comparison
    axes[1, 3].imshow(np.concatenate([original_np, highlighted], axis=1))
    axes[1, 3].set_title('Before | After\n(Score-CAM Highlighting)', 
                         fontsize=11, fontweight='bold')
    axes[1, 3].axis('off')
    
    plt.tight_layout()
    plt.show()

def plot_training_curves(train_losses, val_losses, train_accuracies, val_accuracies):
    """Plot training and validation curves"""
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    epochs = range(1, len(train_losses) + 1)
    ax1.plot(epochs, train_losses, 'b-', label='Training Loss', linewidth=2)
    ax1.plot(epochs, val_losses, 'r-', label='Validation Loss', linewidth=2)
    ax1.set_xlabel('Epoch', fontsize=12)
    ax1.set_ylabel('Loss', fontsize=12)
    ax1.set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
    ax1.legend(fontsize=11)
    ax1.grid(True, alpha=0.3)
    
    ax2.plot(epochs, train_accuracies, 'b-', label='Training Accuracy', linewidth=2)
    ax2.plot(epochs, val_accuracies, 'r-', label='Validation Accuracy', linewidth=2)
    ax2.set_xlabel('Epoch', fontsize=12)
    ax2.set_ylabel('Accuracy (%)', fontsize=12)
    ax2.set_title('Training and Validation Accuracy', fontsize=14, fontweight='bold')
    ax2.legend(fontsize=11)
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# ================== Step 9: Main Execution Pipeline ==================

def main():
    print("="*80)
    print("Enhanced Frequency Domain CNN with Improved Score-CAM - Fruits-360 Dataset")
    print("="*80)
    
    # Dataset path
    data_root = '/kaggle/input/fruits/fruits-360_100x100/fruits-360'
    
    print("\n[Step 1] Loading Fruits-360 dataset...")
    trainset, testset, classes = load_fruits_dataset(data_root)
    print(f"Number of classes: {len(classes)}")
    
    print("\n[Step 2] Splitting training set into train/validation...")
    train_size = int(0.85 * len(trainset))
    val_size = len(trainset) - train_size
    train_subset, val_subset = torch.utils.data.random_split(
        trainset, [train_size, val_size],
        generator=torch.Generator().manual_seed(42)
    )
    
    print(f"Training samples: {len(train_subset)}")
    print(f"Validation samples: {len(val_subset)}")
    print(f"Test samples: {len(testset)}")
    
    print("\n[Step 3] Converting to frequency domain using FFT...")
    freq_train_dataset = FrequencyDomainDataset(train_subset)
    freq_val_dataset = FrequencyDomainDataset(val_subset)
    freq_test_dataset = FrequencyDomainDataset(testset)
    
    train_loader = DataLoader(freq_train_dataset, batch_size=64, shuffle=True, 
                             num_workers=4, pin_memory=True)
    val_loader = DataLoader(freq_val_dataset, batch_size=64, shuffle=False,
                           num_workers=4, pin_memory=True)
    test_loader = DataLoader(freq_test_dataset, batch_size=1, shuffle=False)
    
    print("\n[Step 4] Initializing ResNet50 model for frequency domain...")
    model = FrequencyDomainCNN(num_classes=len(classes), dropout_rate=0.5).to(device)
    print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
    
    print("\n[Step 5] Training model on frequency domain data...")
    train_losses, val_losses, train_accuracies, val_accuracies = train_model(
        model, train_loader, val_loader, 
        epochs=10,
        lr=0.001,
        weight_decay=5e-4
    )
    
    print("\n[Step 5.1] Plotting training curves...")
    plot_training_curves(train_losses, val_losses, train_accuracies, val_accuracies)
    
    print("\n[Step 6] Evaluating on test set...")
    model.eval()
    correct = 0
    total = 0
    
    all_predictions = []
    all_labels = []
    
    with torch.no_grad():
        for freq_images, labels, _ in tqdm(test_loader, desc="Testing"):
            freq_images, labels = freq_images.to(device), labels.to(device)
            outputs = model(freq_images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    test_accuracy = 100 * correct / total
    print(f"\nFinal Test Accuracy: {test_accuracy:.2f}%")
    
    print("\n[Step 7] Applying Enhanced Score-CAM and mapping to spatial domain...")
    print("Generating CIFAR-10 quality visualizations...\n")
    
    # Test on diverse samples
    np.random.seed(42)
    test_indices = np.random.choice(len(testset), min(10, len(testset)), replace=False)
    
    for idx in test_indices:
        original_image, true_label = testset[idx]
        
        freq_features, phase, _ = spatial_to_frequency(original_image.unsqueeze(0))
        freq_input = freq_features.to(device)
        
        # Get model prediction
        model.eval()
        with torch.no_grad():
            output = model(freq_input)
            probabilities = F.softmax(output, dim=1)
            confidence, predicted = torch.max(probabilities.data, 1)
            predicted_class = predicted.item()
            confidence = confidence.item() * 100
        
        print(f"Sample {idx}:")
        print(f"  True Label: {classes[true_label]}")
        print(f"  Predicted: {classes[predicted_class]} ({confidence:.1f}% confidence)")
        
        # Apply improved Score-CAM mapping (CIFAR-10 quality)
        _, saliency_map, highlighted, original_np, cam_freq = apply_improved_scorecam_mapping(
            model, freq_input, phase.squeeze(0), predicted_class, original_image
        )
        
        # Visualize results
        plot_scorecam_results(
            original_np,
            freq_features, 
            cam_freq,
            saliency_map, 
            highlighted,
            predicted_class, 
            true_label, 
            classes,
            confidence
        )
        
        # Clear GPU cache
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        print()
    
    print("="*80)
    print("Pipeline completed successfully!")
    print(f"Final Test Accuracy: {test_accuracy:.2f}%")
    print(f"Total Classes: {len(classes)}")
    print("="*80)
    print("\nKey Improvements Applied (CIFAR-10 Quality):")
    print("✓ FIXED: Lower threshold (40 vs 70) for better object coverage")
    print("✓ FIXED: Minimal morphological operations (less aggressive)")
    print("✓ FIXED: Better Score-CAM normalization matching CIFAR-10")
    print("✓ FIXED: Edge-preserving smoothing instead of guided filter")
    print("✓ FIXED: Proper saliency map generation with correct masking")
    print("✓ FIXED: Smooth alpha blending for natural overlays")
    print("="*80)
    
    # Save the model
    print("\n[Step 8] Saving trained model...")
    torch.save({
        'model_state_dict': model.state_dict(),
        'test_accuracy': test_accuracy,
        'classes': classes,
        'num_classes': len(classes)
    }, 'fruits_fdcnn_improved_scorecam_model.pth')
    print("Model saved as 'fruits_fdcnn_improved_scorecam_model.pth'")

if __name__ == "__main__":
    main()